# Vérification DuckDB ↔ MinIO ↔ Postgres (C19)

Reproduit, avec sortie réellement exécutée, les vérifications menées lors de la décision du mécanisme de jointure du data lake OMEGA LAKE — complément interactif à
[`docs/architecture/integration_infrastructure_omega_lake.md`](../docs/architecture/integration_infrastructure_omega_lake.md)
(§1.3, §1.3bis, §2).

**Prérequis** : `docker compose -f infra/docker/docker-compose.yml up -d db minio minio-init`
(base de staging peuplée par C11, bucket `omega-lake` créé automatiquement).

Deux points vérifiés ici :
1. Le mécanisme de jointure conçu en C18 §5 (`curated/` du lake ↔ base de staging) fonctionne bien avec un Parquet **réellement stocké dans MinIO**, pas seulement un fichier local — via l'extension `httpfs` de DuckDB, configurée explicitement pour MinIO (endpoint + style d'adressage `path`, MinIO n'étant pas AWS S3).
2. L'ingestion batch (`raw/`) copie les fichiers IoT **fidèlement**, vérifié par somme de contrôle SHA-256 entre chaque source et l'objet MinIO — pas seulement "l'upload n'a pas échoué".

In [1]:
import sys

sys.path.insert(0, "..")  # pour importer datacore si le notebook est lance depuis notebooks/

import hashlib

import duckdb

from datacore.config import (
    MINIO_ROOT_PASSWORD,
    MINIO_ROOT_USER,
    OMEGA_LAKE_BUCKET,
    OMEGA_LAKE_S3_ENDPOINT,
)

con = duckdb.connect()
con.sql("INSTALL httpfs; LOAD httpfs;")
con.sql("INSTALL postgres; LOAD postgres;")

con.sql(f"""
    SET s3_endpoint='{OMEGA_LAKE_S3_ENDPOINT}';
    SET s3_access_key_id='{MINIO_ROOT_USER}';
    SET s3_secret_access_key='{MINIO_ROOT_PASSWORD}';
    SET s3_use_ssl=false;
    SET s3_url_style='path';
""")

print("DuckDB connecte, extensions httpfs/postgres chargees, S3 configure pour MinIO.")

DuckDB connecte, extensions httpfs/postgres chargees, S3 configure pour MinIO.


## 1. Jointure curated (Parquet dans MinIO) ↔ base de staging (Postgres)

Ecrit un Parquet de test dans `omega-lake/staging/`, l'attache à la vraie base de staging en lecture seule, et joint les deux sur la clé naturelle `entrepot` → `entrepots.code` (C18 §5).

In [2]:
CLE_TEST = "staging/capteurs_temperature/date=2026-08-01/part-0.parquet"

con.sql(f"""
    COPY (SELECT * FROM read_csv_auto('../data/raw/iot/capteurs_temperature.csv'))
    TO 's3://{OMEGA_LAKE_BUCKET}/{CLE_TEST}' (FORMAT PARQUET)
""")
print(f"Parquet ecrit dans s3://{OMEGA_LAKE_BUCKET}/{CLE_TEST}")

con.sql("""
    ATTACH 'host=localhost port=5432 dbname=datacore_staging user=datacore password=datacore'
    AS staging (TYPE postgres, READ_ONLY)
""")

resultat = con.sql(f"""
    SELECT e.nom, e.ville, count(*) AS nb_mesures, round(avg(c.temperature_c), 2) AS temp_moyenne
    FROM read_parquet('s3://{OMEGA_LAKE_BUCKET}/{CLE_TEST}') c
    JOIN staging.public.entrepots e ON e.code = c.entrepot
    GROUP BY e.nom, e.ville
    ORDER BY e.nom
""")
resultat.show()

Parquet ecrit dans s3://omega-lake/staging/capteurs_temperature/date=2026-08-01/part-0.parquet
┌──────────────────────────┬───────────┬────────────┬──────────────┐
│           nom            │   ville   │ nb_mesures │ temp_moyenne │
│         varchar          │  varchar  │   int64    │    double    │
├──────────────────────────┼───────────┼────────────┼──────────────┤
│ Entrepot Omega Lille     │ Lille     │        864 │         7.97 │
│ Entrepot Omega Lyon      │ Lyon      │        864 │         7.99 │
│ Entrepot Omega Marseille │ Marseille │        864 │         7.99 │
└──────────────────────────┴───────────┴────────────┴──────────────┘



## 2. Vérification indépendante de DuckDB : le fichier existe-t-il vraiment côté MinIO ?

Le résultat ci-dessus pourrait techniquement sortir juste d'un cache mémoire de la session DuckDB qui vient d'écrire le fichier. Confirmation via `boto3`, un client complètement indépendant de DuckDB, que l'objet existe bien côté serveur MinIO.

In [3]:
import boto3
from botocore.client import Config

s3 = boto3.client(
    "s3",
    endpoint_url=f"http://{OMEGA_LAKE_S3_ENDPOINT}",
    aws_access_key_id=MINIO_ROOT_USER,
    aws_secret_access_key=MINIO_ROOT_PASSWORD,
    config=Config(s3={"addressing_style": "path"}),
)

tete = s3.head_object(Bucket=OMEGA_LAKE_BUCKET, Key=CLE_TEST)
taille = tete["ContentLength"]
print(f"Objet confirme cote MinIO (via boto3, independant de DuckDB) : {taille} octets.")

Objet confirme cote MinIO (via boto3, independant de DuckDB) : 12954 octets.


## 3. Ingestion batch réelle des 4 flux + vérification de fidélité (SHA-256)

Exécute `ingerer_flux_batch()` pour de vrai contre MinIO, puis compare chaque fichier source à l'objet déposé octet pour octet (pas seulement "l'upload n'a pas levé d'erreur").

In [4]:
import datetime
from pathlib import Path

from datacore.storage.lake.ingestion_batch import ingerer_flux_batch

date_test = datetime.date(2026, 9, 22)
cles_batch = ingerer_flux_batch(date_ingestion=date_test, s3=s3)

for cle in cles_batch:
    nom_fichier = cle.rsplit("/", 1)[-1]
    chemin_local = Path("../data/raw/iot") / nom_fichier
    empreinte_locale = hashlib.sha256(chemin_local.read_bytes()).hexdigest()
    objet = s3.get_object(Bucket=OMEGA_LAKE_BUCKET, Key=cle)
    empreinte_distante = hashlib.sha256(objet["Body"].read()).hexdigest()
    statut = "OK identique" if empreinte_locale == empreinte_distante else "DIFFERENT !"
    print(f"{cle}: {statut}")

raw/capteurs_temperature/date=2026-09-22/capteurs_temperature.csv: OK identique
raw/geoloc_flotte/date=2026-09-22/geoloc_flotte.csv: OK identique
raw/camera_comptage/date=2026-09-22/camera_comptage.csv: OK identique
raw/rfid_scans/date=2026-09-22/rfid_scans.json: OK identique


## Nettoyage

Ce notebook ne doit laisser aucune donnée de démonstration dans le bucket (même discipline que les vérifications en ligne de commande de C18/C19) — sinon C20 (catalogue) y trouverait un flux fantôme non intentionnel.

In [5]:
for cle in [CLE_TEST, *cles_batch]:
    s3.delete_object(Bucket=OMEGA_LAKE_BUCKET, Key=cle)

reste = s3.list_objects_v2(Bucket=OMEGA_LAKE_BUCKET)
print(f"Objets restants dans {OMEGA_LAKE_BUCKET} apres nettoyage : {reste.get('KeyCount', 0)}")

Objets restants dans omega-lake apres nettoyage : 0
